# TBot_SA1 策略学习 Notebook

本 notebook 用于理解 **TBot_SA1** 的完整数据流：从 LeRobot 数据集原始字段 → `DataTransformFn` 预处理链 → `forward` / `select_action` 推理。

> **环境要求**：`conda activate tbot_sa1`，并确保 `PYTHONPATH` 包含 `/vla/my_tbot/src`。
>
> **与 PI05 官方模式的差异**：TBot 不使用 `make_pre_post_processors`，而是在 `TBotSA1DatasetConfig.data_transforms` 中定义变换链，由 `TransformedLeRobotDataset` 在 `__getitem__` 时自动执行。

In [1]:
import os
import sys
from pathlib import Path

# 离线模式：只用本地模型/数据集，禁止 HuggingFace 联网
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

WORKSPACE_ROOT = Path("/vla/workspace/my_tbot")
SRC_ROOT = WORKSPACE_ROOT / "src"
MODELS_ROOT = Path("/vla/.models")
DATA_ROOT = Path("/vla/workspace/data")

# 若未 pip install -e 当前仓库，需手动加入 src
src_root_str = str(SRC_ROOT)
if src_root_str not in sys.path:
    sys.path.insert(0, src_root_str)

## 1. 模型加载

TBot 策略类为 `TBotSA1Policy`，checkpoint 目录需包含 `config.json` 和 `model.safetensors`。

**config 中说明了输入/输出字段**（state/action 会被 pad 到 `max_state_dim=32` / `max_action_dim=32`）。

In [2]:
import torch
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.TBot_SA1.modeling_tbot_sa1 import TBotSA1Policy

# ---------- 路径配置（按需修改）----------
MODEL_ID = "/vla/workspace/models/tbot_base"
QWEN3_VL_PATH = "/vla/workspace/models/Qwen3-VL-2B-Instruct"
COSMOS_PATH = "/vla/workspace/models/Cosmos-Tokenizer-CI8x8"
DA3_PATH = "/vla/workspace/models/DA3-LARGE-1.1"
DA3_CODE_ROOT = "/vla/workspace/my_tbot/third_party/Depth-Anything-3"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 从 checkpoint 读取 config，并覆盖为当前机器上的本地路径
policy_cfg = PreTrainedConfig.from_pretrained(MODEL_ID, local_files_only=True)
policy_cfg.qwen3_vl_pretrained_path = QWEN3_VL_PATH
policy_cfg.cosmos_tokenizer_path_or_name = COSMOS_PATH
policy_cfg.da3_model_path_or_name = DA3_PATH
policy_cfg.da3_code_root = DA3_CODE_ROOT
policy_cfg.device = str(device)

policy = TBotSA1Policy.from_pretrained(
    MODEL_ID,
    config=policy_cfg,
    local_files_only=True,
).to(device).eval()

# policy

/vla/.conda/miniconda3/envs/mytbot/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70
[INFO ] using MLP layer as FFN
Loading weights from local directory
Loading weights from local directory


## 2. 数据集处理

TBot 训练使用 `TransformedLeRobotDataset`：底层 `LeRobotDataset` 负责按 `delta_timestamps` 取时序帧，外层 transform 链负责归一化、图像重映射、Qwen3-VL tokenization 等。

In [3]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset,LeRobotDatasetMetadata
from lerobot.datasets.factory import resolve_delta_timestamps # 抽帧：历史帧/未来帧数
DATASET_PATH = "/vla/workspace/data/adjust_bottle/aloha-agilex_clean_50"  
ds_meta = LeRobotDatasetMetadata(DATASET_PATH)
delta_timestamps = resolve_delta_timestamps(policy_cfg, ds_meta)
# policy_cfg中 image_delta_indices=[-15, 0, 15]；action chunk = 50
# 计算得到：
for k in delta_timestamps.keys():
    print(f"{k}: {delta_timestamps[k]}")
raw_ds_with_delta = LeRobotDataset(
    repo_id=DATASET_PATH,
    delta_timestamps=delta_timestamps,
    video_backend="pyav",
)
raw_ds_no_delta = LeRobotDataset(
    repo_id=DATASET_PATH,
    video_backend="pyav",
)
print("raw_ds_no_delta[0]['observation.images.cam_high'].shape: ",raw_ds_no_delta[0]['observation.images.cam_high'].shape)
print("raw_ds_with_delta[0]['observation.images.cam_high'].shape: ",raw_ds_with_delta[0]['observation.images.cam_high'].shape)
raw_ds_with_delta

  warnings.warn(



action: [0.0, 0.03333333333333333, 0.06666666666666667, 0.1, 0.13333333333333333, 0.16666666666666666, 0.2, 0.23333333333333334, 0.26666666666666666, 0.3, 0.3333333333333333, 0.36666666666666664, 0.4, 0.43333333333333335, 0.4666666666666667, 0.5, 0.5333333333333333, 0.5666666666666667, 0.6, 0.6333333333333333, 0.6666666666666666, 0.7, 0.7333333333333333, 0.7666666666666667, 0.8, 0.8333333333333334, 0.8666666666666667, 0.9, 0.9333333333333333, 0.9666666666666667, 1.0, 1.0333333333333334, 1.0666666666666667, 1.1, 1.1333333333333333, 1.1666666666666667, 1.2, 1.2333333333333334, 1.2666666666666666, 1.3, 1.3333333333333333, 1.3666666666666667, 1.4, 1.4333333333333333, 1.4666666666666666, 1.5, 1.5333333333333334, 1.5666666666666667, 1.6, 1.6333333333333333]
observation.images.cam_high: [-0.5, 0.0, 0.5]
observation.images.cam_left_wrist: [-0.5, 0.0, 0.5]
observation.images.cam_right_wrist: [-0.5, 0.0, 0.5]
raw_ds_no_delta[0]['observation.images.cam_high'].shape:  torch.Size([3, 480, 640])
raw

LeRobotDataset({
    Repository ID: '/vla/workspace/data/adjust_bottle/aloha-agilex_clean_50',
    Number of selected episodes: '50',
    Number of selected samples: '7188',
    Features: '['observation.state', 'action', 'observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

### 2.1 Behavior Prompt 数据层原型

从这里开始先搭建 dataloader 层面的 BP 原型。第一步只做结构验证：

1. 当前样本仍由 `raw_ds_with_delta[idx]` 提供，保持 TBot 原本的 `image_delta_indices=[-15,0,15]` 和 `action chunk=50`。
2. `BehaviorPromptLeRobotDataset` 为当前 `idx` 选择另一条 episode 作为 behavior prompt 来源。
3. prompt 轨迹先从 `raw_ds_no_delta` 读取单帧；`prompt_num_chunks = ceil(prompt_episode_length / prompt_action_chunk_size)`，也就是由选中的 prompt 轨迹长度动态推导。
4. 每个 prompt chunk 取一个代表帧作为图像/state，同时复用 `raw_ds_with_delta` 读取该代表帧之后的 action window。
5. `BehaviorPromptTransformFn` 先只作为结构化检查/透传，后续再逐步加入 resize、remap、normalize、pad 和 image-only processor。

In [4]:
from __future__ import annotations

from dataclasses import dataclass
from collections import defaultdict
import math
import random
import torch
from torch.utils.data import Dataset

from lerobot.transforms.core import DataTransformFn, DataDict


BP_PREFIX = "behavior_prompt"


@dataclass
class BehaviorPromptConfig:
    prompt_action_chunk_size: int = 50
    max_prompt_chunks: int | None = None  # 仅作为显存/调试安全上限；None 表示按整条轨迹推导
    same_episode_policy: str = "avoid"  # avoid / allow / forbid
    seed: int = 0


class BehaviorPromptTransformFn(DataTransformFn):
    """BP transform 骨架：当前只检查并透传 current + behavior_prompt 结构。"""

    def __call__(self, data: DataDict) -> DataDict:
        prompt = data[BP_PREFIX]
        required_prompt_keys = ["images", "state", "action", "mask", "source_indices", "num_chunks"]
        missing = [k for k in required_prompt_keys if k not in prompt]
        if missing:
            raise KeyError(f"behavior_prompt missing keys: {missing}")
        return data


class BehaviorPromptLeRobotDataset(Dataset):
    """Notebook 原型：为每个当前样本附加另一条轨迹的 behavior prompt。

    current_ds 使用 delta_timestamps，保持 TBot 原本训练样本结构。
    frame_ds 不使用 delta_timestamps，用于按绝对 frame index 抽取 prompt 单帧。
    """

    def __init__(
        self,
        current_ds: LeRobotDataset,
        frame_ds: LeRobotDataset,
        prompt_cfg: BehaviorPromptConfig,
        transform: DataTransformFn | None = None,
    ):
        self.current_ds = current_ds
        self.frame_ds = frame_ds
        self.prompt_cfg = prompt_cfg
        self.transform = transform or BehaviorPromptTransformFn()
        self.rng = random.Random(prompt_cfg.seed)
        self._episode_to_indices = self._build_episode_to_indices()
        self._task_to_episodes = self._build_task_to_episodes()

    def __len__(self):
        return len(self.current_ds)

    def _to_int(self, value):
        if isinstance(value, torch.Tensor):
            return int(value.item())
        return int(value)

    def _build_episode_to_indices(self):
        episode_to_indices = defaultdict(list)
        # Notebook 原型：遍历 hf_dataset 的轻量元数据列，避免逐帧解码图像。
        for i in range(len(self.frame_ds)):
            row = self.frame_ds.hf_dataset[i]
            episode_idx = self._to_int(row["episode_index"])
            absolute_idx = self._to_int(row["index"])
            episode_to_indices[episode_idx].append(absolute_idx)
        return dict(episode_to_indices)

    def _build_task_to_episodes(self):
        task_to_episodes = defaultdict(set)
        for episode_idx, indices in self._episode_to_indices.items():
            first = self.frame_ds.hf_dataset[indices[0]]
            task_idx = self._to_int(first.get("task_index", 0))
            task_to_episodes[task_idx].add(episode_idx)
        return {task_idx: sorted(episodes) for task_idx, episodes in task_to_episodes.items()}

    def _sample_prompt_episode(self, current_episode_idx: int, current_task_idx: int) -> int:
        candidates = list(self._task_to_episodes.get(current_task_idx, []))
        if not candidates:
            candidates = sorted(self._episode_to_indices.keys())

        if self.prompt_cfg.same_episode_policy in {"avoid", "forbid"}:
            different_episode_candidates = [ep for ep in candidates if ep != current_episode_idx]
            if different_episode_candidates:
                candidates = different_episode_candidates
            elif self.prompt_cfg.same_episode_policy == "forbid":
                raise RuntimeError(
                    f"No different prompt episode for episode={current_episode_idx}, task={current_task_idx}"
                )

        return self.rng.choice(candidates)

    def _resolve_num_chunks(self, trajectory_len: int) -> int:
        num_chunks = max(1, math.ceil(trajectory_len / self.prompt_cfg.prompt_action_chunk_size))
        if self.prompt_cfg.max_prompt_chunks is not None:
            num_chunks = min(num_chunks, self.prompt_cfg.max_prompt_chunks)
        return num_chunks

    def _linspace_indices(self, indices: list[int], num: int) -> list[int]:
        if num <= 0:
            return []
        if len(indices) == 1:
            return [indices[0]] * num
        positions = torch.linspace(0, len(indices) - 1, steps=num).round().to(torch.long).tolist()
        return [indices[pos] for pos in positions]

    def _build_prompt(self, current_sample: DataDict) -> dict:
        current_episode_idx = self._to_int(current_sample["episode_index"])
        current_task_idx = self._to_int(current_sample.get("task_index", 0))
        prompt_episode_idx = self._sample_prompt_episode(current_episode_idx, current_task_idx)
        prompt_episode_indices = self._episode_to_indices[prompt_episode_idx]
        prompt_num_chunks = self._resolve_num_chunks(len(prompt_episode_indices))

        prompt_frame_indices = self._linspace_indices(
            prompt_episode_indices,
            prompt_num_chunks,
        )
        index_to_offset = {absolute_idx: offset for offset, absolute_idx in enumerate(prompt_episode_indices)}
        prompt_frame_offsets = [index_to_offset[idx] for idx in prompt_frame_indices]
        source_time_ratio = torch.tensor(
            [offset / max(1, len(prompt_episode_indices) - 1) for offset in prompt_frame_offsets],
            dtype=torch.float32,
        )

        prompt_frames = [self.frame_ds[idx] for idx in prompt_frame_indices]
        image_keys = list(self.frame_ds.meta.camera_keys)

        prompt_images = {
            key: torch.stack([frame[key] for frame in prompt_frames], dim=0)
            for key in image_keys
        }
        prompt_state = torch.stack([frame["observation.state"] for frame in prompt_frames], dim=0)

        # 每个 prompt chunk 的 action 从 current_ds 复用 LeRobot 的 delta action window 读取。
        # 对 frame index 逐个查询 current_ds，可以得到 (prompt_action_chunk_size, action_dim)。
        prompt_actions = []
        prompt_action_masks = []
        for idx in prompt_frame_indices:
            chunk_sample = self.current_ds[idx]
            prompt_actions.append(chunk_sample["action"])
            prompt_action_masks.append(chunk_sample.get("action_is_pad", torch.zeros(chunk_sample["action"].shape[0], dtype=torch.bool)))
        prompt_action = torch.stack(prompt_actions, dim=0)
        prompt_action_is_pad = torch.stack(prompt_action_masks, dim=0)

        return {
            "images": prompt_images,
            "state": prompt_state,
            "action": prompt_action,
            "action_is_pad": prompt_action_is_pad,
            "mask": torch.ones(prompt_num_chunks, dtype=torch.bool),
            "num_chunks": torch.tensor(prompt_num_chunks, dtype=torch.long),
            "chunk_indices": torch.arange(prompt_num_chunks, dtype=torch.long),
            "prompt_action_chunk_size": torch.tensor(self.prompt_cfg.prompt_action_chunk_size, dtype=torch.long),
            "source_episode_index": torch.tensor(prompt_episode_idx, dtype=torch.long),
            "source_episode_length": torch.tensor(len(prompt_episode_indices), dtype=torch.long),
            "source_indices": torch.tensor(prompt_frame_indices, dtype=torch.long),
            "source_frame_offsets": torch.tensor(prompt_frame_offsets, dtype=torch.long),
            "source_time_ratio": source_time_ratio,
            "task_index": torch.tensor(current_task_idx, dtype=torch.long),
        }

    def __getitem__(self, idx: int) -> DataDict:
        current = dict(self.current_ds[idx])
        current[BP_PREFIX] = self._build_prompt(current)
        return self.transform(current)


bp_cfg = BehaviorPromptConfig(
    prompt_action_chunk_size=policy_cfg.chunk_size,
    max_prompt_chunks=None,
    same_episode_policy="avoid",
    seed=0,
)

bp_raw_ds = BehaviorPromptLeRobotDataset(
    current_ds=raw_ds_with_delta,
    frame_ds=raw_ds_no_delta,
    prompt_cfg=bp_cfg,
    transform=BehaviorPromptTransformFn(),
)

print("len(bp_raw_ds):", len(bp_raw_ds))
print("prompt config:", bp_cfg)
bp_raw_ds

len(bp_raw_ds): 7188
prompt config: BehaviorPromptConfig(prompt_action_chunk_size=50, max_prompt_chunks=None, same_episode_policy='avoid', seed=0)


### 2.1.1 检查 BehaviorPromptLeRobotDataset 输出

当前这一步只验证数据层结构，不做 resize/remap/normalize/pad。`current` 样本来自 `raw_ds_with_delta`，因此仍有 3 帧图像和 50 步 action；`behavior_prompt` 来自另一条 episode，采样点数量由 `ceil(source_episode_length / prompt_action_chunk_size)` 动态决定。每个采样点包含 3 路单帧图像、一个 state，以及一个 50 步 action chunk。

In [5]:
bp_raw_ds[0].keys() 
# ['observation.images.cam_high', 'observation.images.cam_left_wrist', 'observation.images.cam_right_wrist', 
# 'observation.state', 'action', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index', 'action_is_pad', 
# 'observation.images.cam_high_is_pad', 'observation.images.cam_left_wrist_is_pad', 'observation.images.cam_right_wrist_is_pad', 
# 'task', 'robot_type', 'behavior_prompt']
bp_raw_ds[0]['behavior_prompt'].keys() 
# ['images', 'state', 'action', 
# 'action_is_pad', 'mask', 'num_chunks', 'prompt_action_chunk_size', 'source_episode_index', 'source_episode_length', 'source_indices', 'task_index']
bp_raw_ds[0]['behavior_prompt']['action'].shape # [3, 50, 14]
bp_raw_ds[0]['behavior_prompt']['state'].shape # [3, 14]
bp_raw_ds[0]['behavior_prompt']['images']['observation.images.cam_high'].shape # [3, 3, 480, 640]

torch.Size([3, 3, 480, 640])

behavior_prompt一共被切分为3块，每块action为50步 * 14 维，state为14维，图像为3路图像3x480x640分辨率

### 2.1.2 当前样本与 BP 字段 shape

先观察 `bp_raw_ds[0]` 的嵌套结构。后续再在这个结构上加入 BP 专用 transform。

In [6]:
bp_sample = bp_raw_ds[0]
bp_prompt = bp_sample[BP_PREFIX]

source_episode_length = bp_prompt["source_episode_length"].item()
prompt_action_chunk_size = bp_prompt["prompt_action_chunk_size"].item()
expected_num_chunks = math.ceil(source_episode_length / prompt_action_chunk_size)
actual_num_chunks = bp_prompt["num_chunks"].item()

print("current keys:", sorted(k for k in bp_sample.keys() if k != BP_PREFIX))
print("behavior_prompt keys:", sorted(bp_prompt.keys()))
print()

print("=== current sample ===")
for k in [
    "episode_index",
    "frame_index",
    "index",
    "task_index",
    "observation.state",
    "action",
    "action_is_pad",
    "observation.images.cam_high",
    "observation.images.cam_left_wrist",
    "observation.images.cam_right_wrist",
]:
    v = bp_sample.get(k)
    if hasattr(v, "shape"):
        print(f"{k:45s} {tuple(v.shape)} {v.dtype}")
    else:
        print(f"{k:45s} {v}")

print("\n=== behavior prompt ===")
print("source_episode_index:", bp_prompt["source_episode_index"].item())
print("source_episode_length:", source_episode_length)
print("prompt_action_chunk_size:", prompt_action_chunk_size)
print("expected_num_chunks=ceil(length/chunk_size):", expected_num_chunks)
print("actual_num_chunks:", actual_num_chunks)
print("source_indices:", bp_prompt["source_indices"].tolist())
print("mask:", bp_prompt["mask"].tolist())
print("state:", tuple(bp_prompt["state"].shape), bp_prompt["state"].dtype)
print("action:", tuple(bp_prompt["action"].shape), bp_prompt["action"].dtype)
print("action_is_pad:", tuple(bp_prompt["action_is_pad"].shape), bp_prompt["action_is_pad"].dtype)
for k, v in bp_prompt["images"].items():
    print(f"images[{k}]:", tuple(v.shape), v.dtype)

current keys: ['action', 'action_is_pad', 'episode_index', 'frame_index', 'index', 'observation.images.cam_high', 'observation.images.cam_high_is_pad', 'observation.images.cam_left_wrist', 'observation.images.cam_left_wrist_is_pad', 'observation.images.cam_right_wrist', 'observation.images.cam_right_wrist_is_pad', 'observation.state', 'robot_type', 'task', 'task_index', 'timestamp']
behavior_prompt keys: ['action', 'action_is_pad', 'chunk_indices', 'images', 'mask', 'num_chunks', 'prompt_action_chunk_size', 'source_episode_index', 'source_episode_length', 'source_frame_offsets', 'source_indices', 'source_time_ratio', 'state', 'task_index']

=== current sample ===
episode_index                                 () torch.int64
frame_index                                   () torch.int64
index                                         () torch.int64
task_index                                    () torch.int64
observation.state                             (14,) torch.float32
action            

In [7]:
print("当前 BP 原型 transform:")
print("  [0] BehaviorPromptTransformFn  # 结构检查/透传")
print("\n后续将替换为专用 BP transform 链:")
print("  [0] BPPadOrSampleChunksFn  # BP 固定 K=4")
print("  [1] BPResizeImagesWithPadFn  # BP 图像 resize")
print("  [2] BPRemapImageKeyTransformFn  # BP 图像 key 统一")
print("  [3] BPNormalizeTransformFn  # BP state/action normalize")
print("  [4] BPComposeFieldsTransform  # BP state/action compose")
print("  [5] BPPadStateAndActionTransformFn  # BP state/action pad 到 32")
print("  [6] BPImgOnlyQwen3VLTransformFn  # BP image-only Qwen processor")
print("  [7] UnifyBPInputsTransformFn  # current 字段 + behavior_prompt 保留")

当前 BP 原型 transform:
  [0] BehaviorPromptTransformFn  # 结构检查/透传

后续将替换为专用 BP transform 链:
  [0] BPPadOrSampleChunksFn  # BP 固定 K=4
  [1] BPResizeImagesWithPadFn  # BP 图像 resize
  [2] BPRemapImageKeyTransformFn  # BP 图像 key 统一
  [3] BPNormalizeTransformFn  # BP state/action normalize
  [4] BPComposeFieldsTransform  # BP state/action compose
  [5] BPPadStateAndActionTransformFn  # BP state/action pad 到 32
  [6] BPImgOnlyQwen3VLTransformFn  # BP image-only Qwen processor
  [7] UnifyBPInputsTransformFn  # current 字段 + behavior_prompt 保留


In [8]:
# BP 原型样本（尚未做 TBot 原始 transform）
sample = bp_raw_ds[0]
prompt = sample[BP_PREFIX]

print("BP sample top-level keys:", sorted(sample.keys()))
print("\ncurrent tensor shapes:")
for k, v in sample.items():
    if k == BP_PREFIX:
        continue
    if hasattr(v, "shape"):
        print(f"  {k:45s} {tuple(v.shape)}")

print("\nbehavior_prompt tensor shapes:")
for k, v in prompt.items():
    if k == "images":
        for image_key, image_value in v.items():
            print(f"  images[{image_key}]: {tuple(image_value.shape)}")
    elif hasattr(v, "shape"):
        print(f"  {k:45s} {tuple(v.shape)}")
    else:
        print(f"  {k:45s} {v}")

BP sample top-level keys: ['action', 'action_is_pad', 'behavior_prompt', 'episode_index', 'frame_index', 'index', 'observation.images.cam_high', 'observation.images.cam_high_is_pad', 'observation.images.cam_left_wrist', 'observation.images.cam_left_wrist_is_pad', 'observation.images.cam_right_wrist', 'observation.images.cam_right_wrist_is_pad', 'observation.state', 'robot_type', 'task', 'task_index', 'timestamp']

current tensor shapes:
  observation.images.cam_high                   (3, 3, 480, 640)
  observation.images.cam_left_wrist             (3, 3, 480, 640)
  observation.images.cam_right_wrist            (3, 3, 480, 640)
  observation.state                             (14,)
  action                                        (50, 14)
  timestamp                                     ()
  frame_index                                   ()
  episode_index                                 ()
  index                                         ()
  task_index                                    (

### 2.2 组装 BP batch

`default_collate` 可以处理当前这种嵌套 dict：`behavior_prompt.images` 会按相机 key 分别 stack，`behavior_prompt.state/action/mask` 会 stack 到 batch 维。

In [9]:
from torch.utils.data.dataloader import default_collate

batch = default_collate([sample])


def to_device(batch, device):
    out = {}
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(device, non_blocking=True)
        elif isinstance(v, dict):
            out[k] = to_device(v, device)
        else:
            out[k] = v
    return out


batch = to_device(batch, device)
bp_batch_prompt = batch[BP_PREFIX]

print("batch current action shape:", batch["action"].shape)
print("batch current cam_high shape:", batch["observation.images.cam_high"].shape)
print("batch BP state shape:", bp_batch_prompt["state"].shape)
print("batch BP action shape:", bp_batch_prompt["action"].shape)
print("batch BP mask shape:", bp_batch_prompt["mask"].shape)
for image_key, image_value in bp_batch_prompt["images"].items():
    print(f"batch BP images[{image_key}] shape:", image_value.shape)

batch current action shape: torch.Size([1, 50, 14])
batch current cam_high shape: torch.Size([1, 3, 3, 480, 640])
batch BP state shape: torch.Size([1, 3, 14])
batch BP action shape: torch.Size([1, 3, 50, 14])
batch BP mask shape: torch.Size([1, 3])
batch BP images[observation.images.cam_high] shape: torch.Size([1, 3, 3, 480, 640])
batch BP images[observation.images.cam_left_wrist] shape: torch.Size([1, 3, 3, 480, 640])
batch BP images[observation.images.cam_right_wrist] shape: torch.Size([1, 3, 3, 480, 640])


In [10]:
batch

{'observation.images.cam_high': tensor([[[[[0.9176, 0.9176, 0.9176,  ..., 0.9098, 0.9098, 0.9098],
            [0.9176, 0.9137, 0.9176,  ..., 0.9059, 0.9098, 0.9098],
            [0.9137, 0.9137, 0.9137,  ..., 0.9059, 0.9098, 0.9098],
            ...,
            [0.8941, 0.8941, 0.9020,  ..., 0.8941, 0.9059, 0.9137],
            [0.8902, 0.8902, 0.8941,  ..., 0.8863, 0.8863, 0.8941],
            [0.8863, 0.8902, 0.8902,  ..., 0.8824, 0.8824, 0.8824]],
 
           [[0.8549, 0.8549, 0.8549,  ..., 0.8471, 0.8471, 0.8471],
            [0.8549, 0.8510, 0.8549,  ..., 0.8431, 0.8471, 0.8471],
            [0.8510, 0.8510, 0.8510,  ..., 0.8431, 0.8471, 0.8471],
            ...,
            [0.8745, 0.8745, 0.8824,  ..., 0.8745, 0.8863, 0.8941],
            [0.8706, 0.8706, 0.8745,  ..., 0.8667, 0.8667, 0.8745],
            [0.8667, 0.8706, 0.8706,  ..., 0.8627, 0.8627, 0.8627]],
 
           [[0.8471, 0.8471, 0.8471,  ..., 0.8392, 0.8392, 0.8392],
            [0.8471, 0.8431, 0.8471,  ..., 0.

### 2.3 Behavior Prompt 专用 TransformFn 原型

这一节不再把 BP 临时展开去套普通 sample transform，而是写一套专门面向 `behavior_prompt` 结构的 notebook 原型 transform。BP 的图像/state/action 保留 K 维 chunk 结构，直接完成 resize、remap、normalize、pad 和 image-only Qwen3-VL processor；current 样本仍单独走 TBot 原始 transform 链。

In [11]:
from dataclasses import replace
from copy import copy
from transformers.models.qwen3_vl import Qwen3VLProcessor

from lerobot.policies.TBot_SA1.configuration_tbot_sa1 import TBotSA1DatasetConfig
from lerobot.policies.TBot_SA1.transform_tbot_sa1 import (
    Qwen3_VLProcessorTransformFn,
    UnifyTBotSA1InputsTransformFn,
)
from lerobot.transforms.constants import get_image_mapping
from lerobot.transforms.core import (
    compose,
    InjectMissingStateActionTransformFn,
    hydrate_normalize_transform,
    hydrate_compose_field_transform,
    hydrate_delta_action_transform,
    hydrate_remap_image_key_transform,
)
from lerobot.transforms.utils import resize_with_pad
from lerobot.utils.constants import ACTION, OBS_IMAGES, OBS_STATE, OBS_STR, SAMPLE_ACTION_LOSS_MASK


def _to_scalar_int(value) -> int:
    return int(value.item() if isinstance(value, torch.Tensor) else value)


def _normalize_tensor(x: torch.Tensor, stats: dict, mode: str = "mean_std") -> torch.Tensor:
    eps = 1e-6
    if mode == "mean_std":
        mean = torch.from_numpy(stats["mean"]).to(x)
        std = torch.from_numpy(stats["std"]).to(x)
        return (x - mean) / (std + eps)
    if mode == "min_max":
        min_v = torch.from_numpy(stats["min"]).to(x)
        max_v = torch.from_numpy(stats["max"]).to(x)
        return (x - min_v) / (max_v - min_v + eps)
    raise ValueError(f"Unknown normalization mode: {mode}")


def _pad_last_dim(x: torch.Tensor, target_dim: int) -> torch.Tensor:
    if x.shape[-1] >= target_dim:
        return x
    return torch.nn.functional.pad(x, (0, target_dim - x.shape[-1]))


def _pad_first_dim(x: torch.Tensor, target_len: int, *, fill_value=0) -> torch.Tensor:
    if x.shape[0] >= target_len:
        return x
    pad_shape = (target_len - x.shape[0], *x.shape[1:])
    pad = torch.full(pad_shape, fill_value, dtype=x.dtype, device=x.device)
    return torch.cat([x, pad], dim=0)


class BPPadOrSampleChunksFn(DataTransformFn):
    """固定 BP chunk 数：超过 target_num_chunks 时均匀下采样，不足时 padding。"""

    def __init__(self, target_num_chunks: int = 4):
        self.target_num_chunks = target_num_chunks

    def __call__(self, data: DataDict) -> DataDict:
        prompt = dict(data[BP_PREFIX])
        src_len = _to_scalar_int(prompt["num_chunks"])
        select_idx = self._select_indices(src_len)
        valid_count = min(src_len, self.target_num_chunks)
        valid_mask = torch.arange(self.target_num_chunks) < valid_count

        prompt["images"] = {
            key: self._select_and_pad(value, select_idx, valid_count, fill_value=0.0)
            for key, value in prompt["images"].items()
        }
        for key in ["state", "action"]:
            prompt[key] = self._select_and_pad(prompt[key], select_idx, valid_count, fill_value=0.0)
        prompt["action_is_pad"] = self._select_and_pad(
            prompt["action_is_pad"], select_idx, valid_count, fill_value=True
        )

        prompt["mask"] = valid_mask.to(dtype=torch.bool)
        prompt["num_chunks"] = torch.tensor(self.target_num_chunks, dtype=torch.long)
        prompt["chunk_indices"] = torch.arange(self.target_num_chunks, dtype=torch.long)
        prompt["source_indices"] = self._select_and_pad(prompt["source_indices"], select_idx, valid_count, fill_value=-1)
        prompt["source_frame_offsets"] = self._select_and_pad(
            prompt["source_frame_offsets"], select_idx, valid_count, fill_value=-1
        )
        prompt["source_time_ratio"] = self._select_and_pad(
            prompt["source_time_ratio"], select_idx, valid_count, fill_value=0.0
        )
        prompt["candidate_num_chunks"] = torch.tensor(src_len, dtype=torch.long)
        prompt["selected_candidate_indices"] = _pad_first_dim(
            select_idx[: self.target_num_chunks],
            self.target_num_chunks,
            fill_value=-1,
        )
        if valid_count < self.target_num_chunks:
            prompt["selected_candidate_indices"][valid_count:] = -1
        data[BP_PREFIX] = prompt
        return data

    def _select_indices(self, src_len: int) -> torch.Tensor:
        if src_len <= 0:
            return torch.empty(0, dtype=torch.long)
        if src_len <= self.target_num_chunks:
            return torch.arange(src_len, dtype=torch.long)
        return torch.linspace(0, src_len - 1, steps=self.target_num_chunks).round().to(torch.long)

    def _select_and_pad(self, value: torch.Tensor, select_idx: torch.Tensor, valid_count: int, *, fill_value):
        selected = value[select_idx] if select_idx.numel() else value[:0]
        selected = selected[: self.target_num_chunks]
        padded = _pad_first_dim(selected, self.target_num_chunks, fill_value=fill_value)
        if valid_count < self.target_num_chunks:
            padded[valid_count:] = fill_value
        return padded


class BPResizeImagesWithPadFn(DataTransformFn):
    """只 resize behavior_prompt.images，保持 K 维 chunk 结构。"""

    def __init__(self, height: int, width: int, mode: str = "bilinear"):
        self.height = height
        self.width = width
        self.mode = mode

    def __call__(self, data: DataDict) -> DataDict:
        prompt = dict(data[BP_PREFIX])
        prompt["images"] = {
            key: resize_with_pad(value, self.height, self.width, self.mode)
            for key, value in prompt["images"].items()
        }
        data[BP_PREFIX] = prompt
        return data


class BPRemapImageKeyTransformFn(DataTransformFn):
    """只 remap behavior_prompt.images 的相机 key 到 observation.images.image0/1/2。"""

    def __init__(self, dataset: LeRobotDataset):
        self.image_mapping = get_image_mapping(dataset.meta.robot_type, dataset.meta.features)
        self.output_image_keys = [f"{OBS_IMAGES}.image0", f"{OBS_IMAGES}.image1", f"{OBS_IMAGES}.image2"]

    def __call__(self, data: DataDict) -> DataDict:
        prompt = dict(data[BP_PREFIX])
        images = prompt["images"]
        remapped = {}
        for source_key, target_key in self.image_mapping.items():
            if source_key in images:
                remapped[target_key] = images[source_key]

        if not remapped:
            available = ", ".join(sorted(images.keys()))
            expected = ", ".join(sorted(self.image_mapping.keys()))
            raise KeyError(f"No BP image keys matched mapping. expected=[{expected}], available=[{available}]")

        first_image = next(iter(remapped.values()))
        for key in self.output_image_keys:
            if key not in remapped:
                remapped[key] = torch.ones_like(first_image)
        prompt["images"] = remapped
        data[BP_PREFIX] = prompt
        return data


class BPNormalizeTransformFn(DataTransformFn):
    """只 normalize behavior_prompt 的 state/action。"""

    def __init__(self, dataset: LeRobotDataset, selected_keys: list[str] | None = None, mode: str = "mean_std"):
        self.norm_stats = dataset.meta.stats
        self.selected_keys = selected_keys or [OBS_STATE, ACTION]
        self.mode = mode

    def __call__(self, data: DataDict) -> DataDict:
        prompt = dict(data[BP_PREFIX])
        key_map = {OBS_STATE: "state", ACTION: "action"}
        for stat_key in self.selected_keys:
            prompt_key = key_map.get(stat_key)
            if prompt_key is None or prompt_key not in prompt or stat_key not in self.norm_stats:
                continue
            prompt[prompt_key] = _normalize_tensor(prompt[prompt_key], self.norm_stats[stat_key], self.mode)
        data[BP_PREFIX] = prompt
        return data


class BPComposeFieldsTransform(DataTransformFn):
    """按 mapping compose behavior_prompt 的 state/action；当前 ALOHA 单字段时基本透传。"""

    def __init__(self, dataset: LeRobotDataset, mapping: dict[str, list[str]] | None = None):
        self.mapping = mapping or {OBS_STATE: [OBS_STATE], ACTION: [ACTION]}

    def __call__(self, data: DataDict) -> DataDict:
        prompt = dict(data[BP_PREFIX])
        # Notebook 原型当前已经把 BP state/action 收敛成 prompt['state']/prompt['action']。
        # 多字段 robot 固化源码时，可以在这里按 mapping 合并多个 BP state/action 子字段。
        data[BP_PREFIX] = prompt
        return data


class BPPadStateAndActionTransformFn(DataTransformFn):
    """只把 behavior_prompt.state/action 的最后一维 pad 到模型配置维度。"""

    def __init__(self, max_state_dim: int = 32, max_action_dim: int = 32):
        self.max_state_dim = max_state_dim
        self.max_action_dim = max_action_dim

    def __call__(self, data: DataDict) -> DataDict:
        prompt = dict(data[BP_PREFIX])
        prompt["state"] = _pad_last_dim(prompt["state"], self.max_state_dim)
        prompt["action"] = _pad_last_dim(prompt["action"], self.max_action_dim)
        data[BP_PREFIX] = prompt
        return data


class ImgOnlyQwen3VLTransformFn(DataTransformFn):
    """只处理 current observation 的 3 路图像，不追加 task/lang tokens。"""

    def __init__(
        self,
        pretrained_model_name_or_path: str,
        spatial_merge_size: int = 2,
        vision_start_token_id: int = 151652,
        vision_end_token_id: int = 151653,
        image_token_id: int = 151655,
    ):
        self.pretrained_model_name_or_path = pretrained_model_name_or_path
        self.spatial_merge_size = spatial_merge_size
        self.vision_start_token_id = vision_start_token_id
        self.vision_end_token_id = vision_end_token_id
        self.image_token_id = image_token_id
        self.processor = None

    def _ensure_processor(self):
        if self.processor is None:
            self.processor = Qwen3VLProcessor.from_pretrained(self.pretrained_model_name_or_path)
            self.vision_start_token_id = self.processor.vision_start_token_id
            self.vision_end_token_id = self.processor.vision_end_token_id
            self.image_token_id = self.processor.image_token_id

    def __call__(self, data: DataDict) -> DataDict:
        self._ensure_processor()
        input_ids = []
        attention_mask = []
        pixel_values = []
        image_grid_thw = []

        for i in range(3):
            image_key = f"{OBS_IMAGES}.image{i}"
            img_inputs = self.processor.image_processor(
                data[image_key][1],  # 与原 TBot 一致：current 三帧里只取中间当前帧
                do_rescale=False,
            )
            grid = img_inputs.image_grid_thw
            token_count = torch.prod(grid) // self.spatial_merge_size ** 2
            pixel_values.append(img_inputs.pixel_values)
            image_grid_thw.append(grid)

            input_ids += [self.vision_start_token_id] + [self.image_token_id] * token_count + [self.vision_end_token_id]
            is_valid = bool(data[f"{image_key}_mask"].item() if isinstance(data[f"{image_key}_mask"], torch.Tensor) else data[f"{image_key}_mask"])
            attention_mask += [1 if is_valid else 0] * (token_count + 2)

        data[f"{OBS_STR}.pixel_values"] = torch.cat(pixel_values)
        data[f"{OBS_STR}.image_grid_thw"] = torch.cat(image_grid_thw)
        data[f"{OBS_STR}.input_ids"] = torch.tensor(input_ids)
        data[f"{OBS_STR}.attention_mask"] = torch.tensor(attention_mask)
        return data


class BPImgOnlyQwen3VLTransformFn(DataTransformFn):
    """只把 behavior_prompt.images 转成 Qwen3-VL 视觉侧输入，不加入 task/lang tokens。"""

    def __init__(
        self,
        pretrained_model_name_or_path: str,
        image_keys: list[str],
        spatial_merge_size: int = 2,
    ):
        self.pretrained_model_name_or_path = pretrained_model_name_or_path
        self.image_keys = list(image_keys)
        self.spatial_merge_size = spatial_merge_size
        self.processor = None

    def _ensure_processor(self):
        if self.processor is None:
            self.processor = Qwen3VLProcessor.from_pretrained(self.pretrained_model_name_or_path)

    def __call__(self, data: DataDict) -> DataDict:
        self._ensure_processor()
        prompt = dict(data[BP_PREFIX])
        num_chunks = _to_scalar_int(prompt["num_chunks"])

        pixel_values = []
        image_grid_thw = []
        image_token_counts = []
        image_chunk_indices = []
        image_camera_indices = []

        for chunk_idx in range(num_chunks):
            for camera_idx, image_key in enumerate(self.image_keys):
                img_inputs = self.processor.image_processor(
                    prompt["images"][image_key][chunk_idx],
                    do_rescale=False,
                )
                grid = img_inputs.image_grid_thw
                token_count = torch.prod(grid) // self.spatial_merge_size ** 2
                pixel_values.append(img_inputs.pixel_values)
                image_grid_thw.append(grid)
                image_token_counts.append(token_count.to(torch.long))
                image_chunk_indices.append(torch.tensor(chunk_idx, dtype=torch.long))
                image_camera_indices.append(torch.tensor(camera_idx, dtype=torch.long))

        prompt["pixel_values"] = torch.cat(pixel_values, dim=0)
        prompt["image_grid_thw"] = torch.cat(image_grid_thw, dim=0)
        prompt["image_token_counts"] = torch.stack(image_token_counts)
        prompt["image_chunk_indices"] = torch.stack(image_chunk_indices)
        prompt["image_camera_indices"] = torch.stack(image_camera_indices)
        data[BP_PREFIX] = prompt
        return data


class UnifyBPInputsTransformFn(DataTransformFn):
    """整理最终模型输入：标准化 current 字段，同时收敛 behavior_prompt schema。"""

    def __call__(self, data: DataDict) -> DataDict:
        default_action_loss_mask = 0.0 if data.get("robot_type") == "egodex_v" else 1.0
        prompt = data[BP_PREFIX]
        bp_out = {
            "images": prompt["images"],
            "state": prompt["state"],
            "action": prompt["action"],
            "action_is_pad": prompt["action_is_pad"],
            "mask": prompt["mask"],
            "chunk_indices": prompt["chunk_indices"],
            "source_time_ratio": prompt["source_time_ratio"],
            "pixel_values": prompt["pixel_values"],
            "image_grid_thw": prompt["image_grid_thw"],
            "image_token_counts": prompt["image_token_counts"],
            "image_chunk_indices": prompt["image_chunk_indices"],
            "image_camera_indices": prompt["image_camera_indices"],
        }
        return {
            OBS_STATE: data[OBS_STATE],
            ACTION: data[ACTION],
            SAMPLE_ACTION_LOSS_MASK: data.get(
                SAMPLE_ACTION_LOSS_MASK,
                torch.tensor([default_action_loss_mask], dtype=torch.float32),
            ),
            f"{OBS_IMAGES}.image0": data[f"{OBS_IMAGES}.image0"],
            f"{OBS_IMAGES}.image1": data[f"{OBS_IMAGES}.image1"],
            f"{OBS_IMAGES}.image2": data[f"{OBS_IMAGES}.image2"],
            f"{OBS_IMAGES}.image0_mask": data[f"{OBS_IMAGES}.image0_mask"],
            f"{OBS_IMAGES}.image1_mask": data[f"{OBS_IMAGES}.image1_mask"],
            f"{OBS_IMAGES}.image2_mask": data[f"{OBS_IMAGES}.image2_mask"],
            f"{OBS_STR}.pixel_values": data[f"{OBS_STR}.pixel_values"],
            f"{OBS_STR}.image_grid_thw": data[f"{OBS_STR}.image_grid_thw"],
            f"{OBS_STR}.input_ids": data[f"{OBS_STR}.input_ids"],
            f"{OBS_STR}.attention_mask": data[f"{OBS_STR}.attention_mask"],
            BP_PREFIX: bp_out,
        }

In [12]:
def hydrate_current_tbot_transforms(transforms: list[DataTransformFn], dataset: LeRobotDataset) -> list[DataTransformFn]:
    """与 TransformedLeRobotDataset.from_base 保持一致，给 current transform 链补 mapping/stats。"""
    transforms = hydrate_normalize_transform(transforms, dataset)
    transforms = hydrate_compose_field_transform(transforms, dataset)
    transforms = hydrate_delta_action_transform(transforms, dataset)
    transforms = hydrate_remap_image_key_transform(transforms, dataset)
    return transforms


bp_image_keys = [f"{OBS_IMAGES}.image0", f"{OBS_IMAGES}.image1", f"{OBS_IMAGES}.image2"]

bp_only_transforms = [
    BPPadOrSampleChunksFn(target_num_chunks=4),
    BPResizeImagesWithPadFn(
        height=policy_cfg.image_resolution[0],
        width=policy_cfg.image_resolution[1],
    ),
    BPRemapImageKeyTransformFn(raw_ds_with_delta),
    BPNormalizeTransformFn(raw_ds_with_delta),
    BPComposeFieldsTransform(raw_ds_with_delta),
    BPPadStateAndActionTransformFn(
        max_state_dim=policy_cfg.max_state_dim,
        max_action_dim=policy_cfg.max_action_dim,
    ),
    BPImgOnlyQwen3VLTransformFn(
        pretrained_model_name_or_path=QWEN3_VL_PATH,
        image_keys=bp_image_keys,
        spatial_merge_size=2,
    ),
]

# policy_cfg 是模型配置，不含 data_transforms；current 数据变换链来自 dataset config。
# 显式传入 repo_id，保持这里和前面 raw dataset 使用的是同一个数据集。
dataset_cfg = TBotSA1DatasetConfig(
    repo_id=DATASET_PATH,
    qwen3_vl_processor_path=QWEN3_VL_PATH,
)
current_tbot_transforms = [
    copy(t)
    for t in dataset_cfg.data_transforms.inputs
    if not isinstance(t, InjectMissingStateActionTransformFn)
]
current_tbot_transforms = hydrate_current_tbot_transforms(current_tbot_transforms, raw_ds_with_delta)
for i, transform in enumerate(current_tbot_transforms):
    if isinstance(transform, Qwen3_VLProcessorTransformFn):
        current_tbot_transforms[i] = ImgOnlyQwen3VLTransformFn(
            pretrained_model_name_or_path=QWEN3_VL_PATH,
            spatial_merge_size=transform.spatial_merge_size,
        )
current_tbot_transforms[-1] = UnifyBPInputsTransformFn()

bp_transform_chain = compose([*bp_only_transforms, *current_tbot_transforms])

bp_transformed_ds = BehaviorPromptLeRobotDataset(
    current_ds=raw_ds_with_delta,
    frame_ds=raw_ds_no_delta,
    prompt_cfg=bp_cfg,
    transform=bp_transform_chain,
)

print("BP-only transforms:")
for i, t in enumerate(bp_only_transforms):
    print(f"  [{i}] {t.__class__.__name__}")
print("\nCurrent TBot transforms with BP unify:")
for i, t in enumerate(current_tbot_transforms):
    print(f"  [{i}] {t.__class__.__name__}")
print("\nlen(bp_transformed_ds):", len(bp_transformed_ds))

Hydrating transform NormalizeTransformFn with dataset.meta.stats (robot_type=aloha, resolved=aloha) and selected_keys (selected_keys=['observation.state', 'action'])
Hydrating transform ComposeFieldsTransform with mapping (robot_type=aloha, resolved=aloha)
Hydrating transform RemapImageKeyTransformFn with mapping (robot_type=aloha, resolved=aloha)
BP-only transforms:
  [0] BPPadOrSampleChunksFn
  [1] BPResizeImagesWithPadFn
  [2] BPRemapImageKeyTransformFn
  [3] BPNormalizeTransformFn
  [4] BPComposeFieldsTransform
  [5] BPPadStateAndActionTransformFn
  [6] BPImgOnlyQwen3VLTransformFn

Current TBot transforms with BP unify:
  [0] ResizeImagesWithPadFn
  [1] RemapImageKeyTransformFn
  [2] NormalizeTransformFn
  [3] ComposeFieldsTransform
  [4] PadStateAndActionTransformFn
  [5] ImgOnlyQwen3VLTransformFn
  [6] UnifyBPInputsTransformFn

len(bp_transformed_ds): 7188


In [13]:
for i, t in enumerate(bp_transformed_ds.transform.transforms):
    print(i, t.__class__.__name__)

0 BPPadOrSampleChunksFn
1 BPResizeImagesWithPadFn
2 BPRemapImageKeyTransformFn
3 BPNormalizeTransformFn
4 BPComposeFieldsTransform
5 BPPadStateAndActionTransformFn
6 BPImgOnlyQwen3VLTransformFn
7 ResizeImagesWithPadFn
8 RemapImageKeyTransformFn
9 NormalizeTransformFn
10 ComposeFieldsTransform
11 PadStateAndActionTransformFn
12 ImgOnlyQwen3VLTransformFn
13 UnifyBPInputsTransformFn


### 2.3.1 检查专用 BP TransformFn 输出

先用单样本检查完整 transform 链。`BehaviorPromptLeRobotDataset` 仍可产生动态数量的候选 chunks；`BPPadOrSampleChunksFn` 会把它们固定到 `K=4`，超过则均匀下采样，不足则 padding 并写入 `mask`。随后 BP 再完成 resize/remap/normalize/pad 和 image-only Qwen3-VL processor。

In [14]:
bp_t_sample = bp_transformed_ds[0]
bp_t_prompt = bp_t_sample[BP_PREFIX]

print("=== final transformed current sample / 最终模型输入中的当前观测部分 ===")
for key in [
    OBS_STATE,
    ACTION,
    SAMPLE_ACTION_LOSS_MASK,
    f"{OBS_IMAGES}.image0",
    f"{OBS_IMAGES}.image1",
    f"{OBS_IMAGES}.image2",
    f"{OBS_STR}.pixel_values",
    f"{OBS_STR}.image_grid_thw",
    f"{OBS_STR}.input_ids",
    f"{OBS_STR}.attention_mask",
]:
    value = bp_t_sample[key]
    print(f"{key:45s} {tuple(value.shape)} {value.dtype}")

print("\n=== final behavior_prompt schema / unify 后保留给模型侧的 BP 字段 ===")
for key in [
    "mask",
    "chunk_indices",
    "source_time_ratio",
    "state",
    "action",
    "action_is_pad",
    "pixel_values",
    "image_grid_thw",
    "image_token_counts",
    "image_chunk_indices",
    "image_camera_indices",
]:
    value = bp_t_prompt[key]
    if hasattr(value, "shape"):
        print(f"{key:45s} {tuple(value.shape)} {value.dtype}")
    else:
        print(f"{key:45s} {value}")

print("\n=== behavior_prompt images after resize/remap / BP 图像 resize+remap 后 ===")
for key, value in bp_t_prompt["images"].items():
    print(f"images[{key}]: {tuple(value.shape)} {value.dtype}")

current_image_only_len = bp_t_sample[f"{OBS_STR}.input_ids"].shape[0]
current_expected_len = int(((bp_t_sample[f"{OBS_STR}.image_grid_thw"].prod(dim=1) // 4) + 2).sum().item())
print("\ncurrent image-only input_ids length / 当前观测 image-only token 长度:", current_image_only_len)
print("current expected image-only length / 根据 image_grid_thw 计算的期望长度:", current_expected_len)
print("task token length removed / 已去除 task tokenizer 的 48 个文本 token:", current_image_only_len == current_expected_len)

expected_image_entries = bp_t_prompt["mask"].shape[0] * len(bp_image_keys)
print("\nexpected BP image entries / 期望 BP 图像条目数 = K * num_cameras:", expected_image_entries)
print("actual BP image_grid_thw entries / 实际 BP Qwen 图像网格条目数:", bp_t_prompt["image_grid_thw"].shape[0])

=== final transformed current sample / 最终模型输入中的当前观测部分 ===
observation.state                             (32,) torch.float32
action                                        (50, 32) torch.float32
sample.action_loss_mask                       (1,) torch.float32
observation.images.image0                     (3, 3, 224, 224) torch.float32
observation.images.image1                     (3, 3, 224, 224) torch.float32
observation.images.image2                     (3, 3, 224, 224) torch.float32
observation.pixel_values                      (768, 1536) torch.float32
observation.image_grid_thw                    (3, 3) torch.int64
observation.input_ids                         (198,) torch.int64
observation.attention_mask                    (198,) torch.int64

=== final behavior_prompt schema / unify 后保留给模型侧的 BP 字段 ===
mask                                          (4,) torch.bool
chunk_indices                                 (4,) torch.int64
source_time_ratio                             (4,) torch.f

In [15]:
bp_t_batch = default_collate([bp_t_sample])
bp_t_batch = to_device(bp_t_batch, device)
bp_t_batch_prompt = bp_t_batch[BP_PREFIX]

print("=== final transformed batch current / 最终 batch 中的当前观测部分 ===")
for key in [OBS_STATE, ACTION, f"{OBS_STR}.pixel_values", f"{OBS_STR}.image_grid_thw", f"{OBS_STR}.input_ids", f"{OBS_STR}.attention_mask"]:
    value = bp_t_batch[key]
    print(f"{key:45s} {tuple(value.shape)} {value.dtype} {value.device}")

print("\n=== final batch[behavior_prompt] schema / unify 后保留给模型侧的 BP batch 字段 ===")
for key in [
    "state",
    "action",
    "action_is_pad",
    "mask",
    "chunk_indices",
    "source_time_ratio",
    "pixel_values",
    "image_grid_thw",
    "image_token_counts",
    "image_chunk_indices",
    "image_camera_indices",
]:
    value = bp_t_batch_prompt[key]
    print(f"{key:45s} {tuple(value.shape)} {value.dtype} {value.device}")

current_batch_image_only_len = bp_t_batch[f"{OBS_STR}.input_ids"].shape[1]
current_batch_expected_len = int(((bp_t_batch[f"{OBS_STR}.image_grid_thw"][0].prod(dim=1) // 4) + 2).sum().item())
print("\nCurrent batch image-only input_ids length / 当前 batch image-only token 长度:", current_batch_image_only_len)
print("Current batch expected image-only length / 根据 image_grid_thw 计算的期望长度:", current_batch_expected_len)
print("Task tokens removed from current input_ids / 当前 input_ids 已去掉 task token:", current_batch_image_only_len == current_batch_expected_len)

print("\nBP batch fixed K / 固定后的 BP chunk 数:", bp_t_batch_prompt["mask"].shape[1])
print("BP batch valid mask / BP 有效 chunk mask:", bp_t_batch_prompt["mask"].tolist())

=== final transformed batch current / 最终 batch 中的当前观测部分 ===
observation.state                             (1, 32) torch.float32 cuda:0
action                                        (1, 50, 32) torch.float32 cuda:0
observation.pixel_values                      (1, 768, 1536) torch.float32 cuda:0
observation.image_grid_thw                    (1, 3, 3) torch.int64 cuda:0
observation.input_ids                         (1, 198) torch.int64 cuda:0
observation.attention_mask                    (1, 198) torch.int64 cuda:0

=== final batch[behavior_prompt] schema / unify 后保留给模型侧的 BP batch 字段 ===
state                                         (1, 4, 32) torch.float32 cuda:0
action                                        (1, 4, 50, 32) torch.float32 cuda:0
action_is_pad                                 (1, 4, 50) torch.bool cuda:0
mask                                          (1, 4) torch.bool cuda:0
chunk_indices                                 (1, 4) torch.int64 cuda:0
source_time_ratio            

### TransformFn Step-by-Step / 逐步查看每个 TransformFn 的作用

从无 transform 的 `bp_ds[0]` 开始，按 `bp_transform_chain` 的真实顺序逐个执行 TransformFn。每一步打印当前样本 top-level 字段和 `behavior_prompt` 子字段的关键 shape，用于观察该 TransformFn 修改了哪些数据。

In [16]:
bp_ds = BehaviorPromptLeRobotDataset(
    current_ds=raw_ds_with_delta,
    frame_ds=raw_ds_no_delta,
    prompt_cfg=bp_cfg,
)

In [17]:
def clone_nested_for_debug(value):
    if isinstance(value, torch.Tensor):
        return value.clone()
    if isinstance(value, dict):
        return {k: clone_nested_for_debug(v) for k, v in value.items()}
    return value


def shape_desc(value):
    if isinstance(value, torch.Tensor):
        return f"{tuple(value.shape)} {value.dtype}"
    if isinstance(value, dict):
        return f"dict[{len(value)}]"
    return repr(value)


def print_current_summary(sample: DataDict):
    print("  current/top-level / 当前观测顶层字段:")
    for key in [
        OBS_STATE,
        ACTION,
        "action_is_pad",
        SAMPLE_ACTION_LOSS_MASK,
        "observation.images.cam_high",
        "observation.images.cam_left_wrist",
        "observation.images.cam_right_wrist",
        f"{OBS_IMAGES}.image0",
        f"{OBS_IMAGES}.image1",
        f"{OBS_IMAGES}.image2",
        f"{OBS_STR}.pixel_values",
        f"{OBS_STR}.image_grid_thw",
        f"{OBS_STR}.input_ids",
        f"{OBS_STR}.attention_mask",
    ]:
        if key in sample:
            print(f"    {key:45s} {shape_desc(sample[key])}")


def print_bp_summary(sample: DataDict):
    if BP_PREFIX not in sample:
        print("  behavior_prompt: <missing>")
        return
    prompt = sample[BP_PREFIX]
    print("  behavior_prompt / BP 子结构:")
    for key in [
        "num_chunks",
        "candidate_num_chunks",
        "prompt_action_chunk_size",
        "source_episode_length",
        "source_episode_index",
        "selected_candidate_indices",
        "chunk_indices",
        "source_indices",
        "source_frame_offsets",
        "source_time_ratio",
        "mask",
        "state",
        "action",
        "action_is_pad",
        "pixel_values",
        "image_grid_thw",
        "image_token_counts",
        "image_chunk_indices",
        "image_camera_indices",
    ]:
        if key in prompt:
            print(f"    {key:45s} {shape_desc(prompt[key])}")
    if "images" in prompt:
        for image_key, image_value in prompt["images"].items():
            print(f"    images[{image_key}] {shape_desc(image_value)}")


def print_sample_summary(title: str, sample: DataDict):
    print("\n" + "=" * 100)
    print(title)
    print_current_summary(sample)
    print_bp_summary(sample)


step_transforms = list(bp_only_transforms) + list(current_tbot_transforms)
print("Transform chain / 实际执行顺序:")
for i, transform in enumerate(step_transforms):
    print(f"  {i:02d}. {transform.__class__.__name__}")

step_sample = clone_nested_for_debug(bp_ds[0])
print_sample_summary("step -1: raw bp_ds[0] / 无 TransformFn 的原始 BP 样本", step_sample)

for i, transform in enumerate(step_transforms):
    before_keys = set(step_sample.keys())
    step_sample = transform(step_sample)
    after_keys = set(step_sample.keys())
    added = sorted(after_keys - before_keys)
    removed = sorted(before_keys - after_keys)
    title = f"step {i}: {transform.__class__.__name__}"
    if added or removed:
        title += f" | added={added} removed={removed}"
    print_sample_summary(title, step_sample)

Transform chain / 实际执行顺序:
  00. BPPadOrSampleChunksFn
  01. BPResizeImagesWithPadFn
  02. BPRemapImageKeyTransformFn
  03. BPNormalizeTransformFn
  04. BPComposeFieldsTransform
  05. BPPadStateAndActionTransformFn
  06. BPImgOnlyQwen3VLTransformFn
  07. ResizeImagesWithPadFn
  08. RemapImageKeyTransformFn
  09. NormalizeTransformFn
  10. ComposeFieldsTransform
  11. PadStateAndActionTransformFn
  12. ImgOnlyQwen3VLTransformFn
  13. UnifyBPInputsTransformFn

step -1: raw bp_ds[0] / 无 TransformFn 的原始 BP 样本
  current/top-level / 当前观测顶层字段:
    observation.state                             (14,) torch.float32
    action                                        (50, 14) torch.float32
    action_is_pad                                 (50,) torch.bool
    observation.images.cam_high                   (3, 3, 480, 640) torch.float32
    observation.images.cam_left_wrist             (3, 3, 480, 640) torch.float32
    observation.images.cam_right_wrist            (3, 3, 480, 640) torch.float32
  beh

## 3. BP Forward 原型

这一节是当前唯一保留的 BP forward 主线：从 `bp_t_batch` 出发，显式构造 BP prefix，并完整复刻 `policy.forward` 中的 `loss_action`、`loss_gen`、`loss_3d`。

核心设计：

1. current 图像和 BP 图像一起进入 `policy.model.qwen3_vl_with_expert.und_expert.visual(...)`。
2. current/BP 图像 token 加 `type_embedding` 区分来源。
3. BP 图像 token 和 state-action prompt token 加 `chunk_embedding` 表达 4 个 prompt chunk 的轨迹顺序。
4. BP action 保留完整 `(B, K, 50, 32)`，加入 step position 后 flatten 为 `(B, K, 1600)`，再经 MLP 映射到 `(B, K, 2048)`。
5. BP state 映射到 `(B, K, 2048)` 后与 action embedding 融合，得到最终 BP state-action prompt token。

In [18]:
import torch.nn as nn
import torch.nn.functional as F


def repeat_segments_by_counts(segments: list[torch.Tensor], image_token_counts: torch.Tensor, camera_indices: torch.Tensor) -> torch.Tensor:
    """按 BP 的 camera index 复用 current image-only token 模板。

    image_token_counts 是纯 image placeholder 数，不包含 Qwen3-VL 的 vision_start / vision_end。
    因此每张图在 input_ids 里的实际长度是 image_token_count + 2。
    """
    repeated = []
    for image_token_count, camera_idx in zip(image_token_counts.tolist(), camera_indices.tolist(), strict=False):
        segment = segments[int(camera_idx)]
        expected_len = int(image_token_count) + 2
        if segment.numel() != expected_len:
            raise ValueError(
                f"Token template length mismatch for camera={camera_idx}: "
                f"template={segment.numel()}, expected={expected_len} (= image_token_count {int(image_token_count)} + 2)"
            )
        repeated.append(segment)
    return torch.cat(repeated, dim=0)


def split_current_image_only_segments(current_input_ids: torch.Tensor, current_image_grid_thw: torch.Tensor) -> list[torch.Tensor]:
    # 每张图 token 数 = image placeholder 数 + vision_start/vision_end 两个边界 token。
    counts = (current_image_grid_thw.prod(dim=-1) // 4 + 2).to(torch.long)
    segments = []
    cursor = 0
    for count in counts.tolist():
        segments.append(current_input_ids[cursor : cursor + int(count)])
        cursor += int(count)
    if cursor != current_input_ids.numel():
        raise ValueError(f"current input_ids length mismatch: split={cursor}, actual={current_input_ids.numel()}")
    return segments


def build_bp_visual_chunk_token_indices(bp_prompt: dict[str, torch.Tensor]) -> torch.Tensor:
    # 展开到 token-level；每张图额外包含 vision_start / vision_end 两个边界 token。
    token_counts = bp_prompt["image_token_counts"].to(torch.long) + 2
    image_chunk_indices = bp_prompt["image_chunk_indices"].to(torch.long)
    return torch.repeat_interleave(image_chunk_indices, token_counts, dim=0)

print("========== 4.1 explicit BP prefix / 显式构造 BP prefix ==========")

bp2_prompt = bp_t_batch[BP_PREFIX]
bp2_image_token_id = policy.model.qwen3_vl_with_expert.und_expert.config.image_token_id
bp2_hidden_size = policy.model.qwen3_vl_with_expert.und_expert.config.text_config.hidden_size
bp2_num_chunks = bp2_prompt["mask"].shape[1]

# 1) 合并 current 图像与 BP 图像。这里对应你的理解：图像一起送进 Qwen3-VL visual encoder。
bp2_pixel_values = torch.cat(
    [bp_t_batch[f"{OBS_STR}.pixel_values"], bp2_prompt["pixel_values"]],
    dim=1,
)
bp2_image_grid_thw = torch.cat(
    [bp_t_batch[f"{OBS_STR}.image_grid_thw"], bp2_prompt["image_grid_thw"]],
    dim=1,
)

# 2) 构造 current+BP 的 image-only input_ids。
# current input_ids 已经有 3 张图的 vision_start / image tokens / vision_end。
# BP 这里复用 current 三路相机的 image-only 模板，每个 BP chunk 有 3 张图。
bp2_current_ids = bp_t_batch[f"{OBS_STR}.input_ids"]
bp2_current_mask = bp_t_batch[f"{OBS_STR}.attention_mask"]
bp2_bp_ids_list = []
bp2_bp_chunk_token_ids_list = []
for b in range(bp2_current_ids.shape[0]):
    current_segments = split_current_image_only_segments(
        bp2_current_ids[b],
        bp_t_batch[f"{OBS_STR}.image_grid_thw"][b],
    )
    bp_ids_b = repeat_segments_by_counts(
        current_segments,
        bp2_prompt["image_token_counts"][b],
        bp2_prompt["image_camera_indices"][b],
    )
    bp2_bp_ids_list.append(bp_ids_b)
    bp2_bp_chunk_token_ids_list.append(
        build_bp_visual_chunk_token_indices(
            {
                "image_token_counts": bp2_prompt["image_token_counts"][b],
                "image_chunk_indices": bp2_prompt["image_chunk_indices"][b],
            }
        )
    )

bp2_bp_ids = torch.stack(bp2_bp_ids_list, dim=0).to(device=bp2_current_ids.device)
bp2_bp_chunk_token_ids = torch.stack(bp2_bp_chunk_token_ids_list, dim=0).to(device=bp2_current_ids.device)
bp2_input_ids = torch.cat([bp2_current_ids, bp2_bp_ids], dim=1)
bp2_attention_mask = torch.ones_like(bp2_input_ids, dtype=bp2_current_mask.dtype)

# 3) 这几行就是原 embed_prefix 的核心：visual encoder -> token embedding -> scatter image embedding。
bp2_D_pixel = bp2_pixel_values.shape[-1]
bp2_image_embs, _ = policy.model.qwen3_vl_with_expert.und_expert.visual(
    bp2_pixel_values.view(-1, bp2_D_pixel),
    bp2_image_grid_thw.view(-1, 3),
)
bp2_prefix_embs = policy.model.qwen3_vl_with_expert.und_expert.get_input_embeddings()(bp2_input_ids)
bp2_B, bp2_L, bp2_D = bp2_prefix_embs.shape
bp2_prefix_embs_flat = bp2_prefix_embs.view(-1, bp2_D)
bp2_input_ids_flat = bp2_input_ids.view(-1)
bp2_prefix_embs_flat[bp2_input_ids_flat == bp2_image_token_id] = bp2_image_embs
bp2_visual_prefix_embs = bp2_prefix_embs_flat.view(bp2_B, bp2_L, bp2_D)

# 4) pos / type 信息集中在这里：
# - type_embedding: current=0, BP=1，用来让模型知道 token 来源。
# - chunk_embedding: 只给 BP token，用来表达第几个 behavior chunk。
bp2_type_embedding = nn.Embedding(2, bp2_hidden_size).to(device=device, dtype=torch.float32)
bp2_chunk_embedding = nn.Embedding(bp2_num_chunks, bp2_hidden_size).to(device=device, dtype=torch.float32)
bp2_current_len = bp2_current_ids.shape[1]
bp2_bp_visual_len = bp2_bp_ids.shape[1]

bp2_current_type_ids = torch.zeros((bp2_B, bp2_current_len), dtype=torch.long, device=device)
bp2_bp_type_ids = torch.ones((bp2_B, bp2_bp_visual_len), dtype=torch.long, device=device)
bp2_current_visual_embs = bp2_visual_prefix_embs[:, :bp2_current_len]
bp2_bp_visual_embs = bp2_visual_prefix_embs[:, bp2_current_len:]
bp2_current_visual_embs = bp2_current_visual_embs + bp2_type_embedding(bp2_current_type_ids).to(dtype=bp2_current_visual_embs.dtype)
bp2_bp_visual_embs = bp2_bp_visual_embs + bp2_type_embedding(bp2_bp_type_ids).to(dtype=bp2_bp_visual_embs.dtype)
bp2_bp_visual_embs = bp2_bp_visual_embs + bp2_chunk_embedding(bp2_bp_chunk_token_ids).to(dtype=bp2_bp_visual_embs.dtype)

for name, value in [
    ("bp2_image_embs from visual encoder", bp2_image_embs),
    ("bp2_visual_prefix_embs before pos", bp2_visual_prefix_embs),
    ("bp2_current_visual_embs after type pos", bp2_current_visual_embs),
    ("bp2_bp_visual_embs after type+chunk pos", bp2_bp_visual_embs),
    ("bp2_input_ids", bp2_input_ids),
    ("bp2_image_grid_thw", bp2_image_grid_thw),
]:
    print(f"{name:48s} {tuple(value.shape)} {value.dtype} {value.device}")

========== 4.1 explicit BP prefix / 显式构造 BP prefix ==========
bp2_image_embs from visual encoder               (960, 2048) torch.bfloat16 cuda:0
bp2_visual_prefix_embs before pos                (1, 990, 2048) torch.bfloat16 cuda:0
bp2_current_visual_embs after type pos           (1, 198, 2048) torch.bfloat16 cuda:0
bp2_bp_visual_embs after type+chunk pos          (1, 792, 2048) torch.bfloat16 cuda:0
bp2_input_ids                                    (1, 990) torch.int64 cuda:0
bp2_image_grid_thw                               (1, 15, 3) torch.int64 cuda:0


In [19]:
print("========== 4.2 state/action prompt tokens / 显式构造 state-action prompt token ==========")

# state/action 的关系：
# - state 是每个 BP chunk 的代表帧状态，先由 state_mlp 投影成 (B, K, hidden)。
# - action 保留完整 50 step x 32 dim 结构，先给每个 step 加显式 step position，再 flatten 成 (B, K, 50*32)。
# - flatten 后经过多层 action_mlp 映射成 (B, K, hidden)，避免 mean pooling 直接压掉细粒度动作信息。
# - action_is_pad=True 的 step 会先置零；step position 也只加在有效 step 上，避免 padding step 产生伪信息。
# - state/action 各自得到 hidden token 后再 cat + fuse_mlp，得到每个 chunk 1 个 state-action prompt token。
# - chunk_embedding(prompt["chunk_indices"]) 表达 4 个 BP chunk 在整条 prompt 轨迹中的顺序，不表达 50 个 step 的块内顺序。

bp2_state = bp2_prompt["state"]                         # (B, K, state_dim)
bp2_action = bp2_prompt["action"]                       # (B, K, 50, action_dim)
bp2_action_is_pad = bp2_prompt.get("action_is_pad")      # (B, K, 50), True 表示 padding step

bp2_state_dim = bp2_state.shape[-1]
bp2_action_chunk_size = bp2_action.shape[-2]
bp2_action_dim = bp2_action.shape[-1]
bp2_action_flat_dim = bp2_action_chunk_size * bp2_action_dim

bp2_state_mlp = nn.Sequential(
    nn.LayerNorm(bp2_state_dim),
    nn.Linear(bp2_state_dim, bp2_hidden_size),
    nn.SiLU(),
    nn.Linear(bp2_hidden_size, bp2_hidden_size),
).to(device=device, dtype=torch.float32)

# action_step_embedding 是 (50, 32)，直接加到每个 action step 的 32 维上。
bp2_action_step_embedding = nn.Embedding(bp2_action_chunk_size, bp2_action_dim).to(device=device, dtype=torch.float32)

# action_mlp 接收完整 50*32=1600 维动作序列，输出一个 chunk-level action token。
bp2_action_mlp = nn.Sequential(
    nn.LayerNorm(bp2_action_flat_dim),
    nn.Linear(bp2_action_flat_dim, bp2_hidden_size * 2),
    nn.SiLU(),
    nn.Linear(bp2_hidden_size * 2, bp2_hidden_size * 2),
    nn.SiLU(),
    nn.Linear(bp2_hidden_size * 2, bp2_hidden_size),
).to(device=device, dtype=torch.float32)

bp2_state_action_fuse_mlp = nn.Sequential(
    nn.LayerNorm(bp2_hidden_size * 2),
    nn.Linear(bp2_hidden_size * 2, bp2_hidden_size),
    nn.SiLU(),
    nn.Linear(bp2_hidden_size, bp2_hidden_size),
).to(device=device, dtype=torch.float32)

# 1) state: (B, K, state_dim) -> (B, K, hidden)
bp2_state_embs = bp2_state_mlp(bp2_state)

# 2) action pad mask: padding step 不参与 action 表示。
if bp2_action_is_pad is None:
    bp2_action_valid_mask = torch.ones(bp2_action.shape[:3], dtype=torch.bool, device=bp2_action.device)
else:
    bp2_action_valid_mask = ~bp2_action_is_pad.to(dtype=torch.bool, device=bp2_action.device)

# 3) action step position: (50, 32)，每个 step 的 32 维共享该 step 的位置向量。
bp2_action_step_ids = torch.arange(bp2_action.shape[2], device=bp2_action.device)
bp2_action_step_pos = bp2_action_step_embedding(bp2_action_step_ids).to(dtype=bp2_action.dtype)
bp2_action_with_step_pos = bp2_action + bp2_action_step_pos.view(1, 1, bp2_action.shape[2], bp2_action.shape[3])
bp2_action_with_step_pos = bp2_action_with_step_pos * bp2_action_valid_mask.unsqueeze(-1)

# 4) action flatten: (B, K, 50, 32) -> (B, K, 1600)，保留每 32 维为一个 step group 的结构。
bp2_action_flat = bp2_action_with_step_pos.flatten(start_dim=2)
bp2_action_embs = bp2_action_mlp(bp2_action_flat)

# 5) state/action fusion: (B, K, hidden) + (B, K, hidden) -> (B, K, hidden)
bp2_state_action_embs = torch.cat([bp2_state_embs, bp2_action_embs], dim=-1)
bp2_state_action_prompt_embs = bp2_state_action_fuse_mlp(bp2_state_action_embs)

# 6) chunk position: 给每个 BP chunk 的 state-action prompt token 加轨迹块位置。
bp2_state_action_prompt_embs = bp2_state_action_prompt_embs + bp2_chunk_embedding(
    bp2_prompt["chunk_indices"].to(device=bp2_state_action_prompt_embs.device)
).to(dtype=bp2_state_action_prompt_embs.dtype)
bp2_state_action_prompt_embs = bp2_state_action_prompt_embs.to(dtype=bp2_current_visual_embs.dtype)

# 7) 拼成新的 prefix：current visual + BP visual + BP state/action prompt。
bp2_prefix_embs = torch.cat(
    [bp2_current_visual_embs, bp2_bp_visual_embs, bp2_state_action_prompt_embs],
    dim=1,
)
bp2_prefix_pad_masks = torch.cat(
    [
        bp2_attention_mask.to(dtype=torch.bool, device=device),
        bp2_prompt["mask"].to(dtype=torch.bool, device=device),
    ],
    dim=1,
)
bp2_prefix_att_masks = torch.zeros_like(bp2_prefix_pad_masks, dtype=torch.bool, device=device)

# get_position_ids 需要 lang token 序列长度覆盖 prefix；state/action prompt 用 pseudo token 占位。
bp2_pseudo_token_id = 777
bp2_prompt_token_ids = torch.full(
    bp2_prompt["mask"].shape,
    fill_value=bp2_pseudo_token_id,
    dtype=bp2_input_ids.dtype,
    device=bp2_input_ids.device,
)
bp2_prefix_lang_tokens = torch.cat([bp2_input_ids, bp2_prompt_token_ids], dim=1)

for name, value in [
    ("bp2_state", bp2_state),
    ("bp2_action", bp2_action),
    ("bp2_state_embs", bp2_state_embs),
    ("bp2_action_step_pos", bp2_action_step_pos),
    ("bp2_action_with_step_pos", bp2_action_with_step_pos),
    ("bp2_action_flat", bp2_action_flat),
    ("bp2_action_embs", bp2_action_embs),
    ("bp2_state_action_embs", bp2_state_action_embs),
    ("bp2_state_action_prompt_embs", bp2_state_action_prompt_embs),
    ("bp2_prefix_embs", bp2_prefix_embs),
    ("bp2_prefix_pad_masks", bp2_prefix_pad_masks),
    ("bp2_prefix_att_masks", bp2_prefix_att_masks),
    ("bp2_prefix_lang_tokens", bp2_prefix_lang_tokens),
]:
    print(f"{name:48s} {tuple(value.shape)} {value.dtype} {value.device}")

print("prefix length breakdown / prefix 长度拆解:")
print("  current visual tokens:", bp2_current_visual_embs.shape[1])
print("  BP visual tokens     :", bp2_bp_visual_embs.shape[1])
print("  BP state/action tokens:", bp2_state_action_prompt_embs.shape[1])
print("  total prefix tokens  :", bp2_prefix_embs.shape[1])
print("action flat dim / action flatten 后维度:", bp2_action_flat_dim)
print("action step pos shape / action 块内 step position:", tuple(bp2_action_step_embedding.weight.shape))

========== 4.2 state/action prompt tokens / 显式构造 state-action prompt token ==========
bp2_state                                        (1, 4, 32) torch.float32 cuda:0
bp2_action                                       (1, 4, 50, 32) torch.float32 cuda:0
bp2_state_embs                                   (1, 4, 2048) torch.float32 cuda:0
bp2_action_step_pos                              (50, 32) torch.float32 cuda:0
bp2_action_with_step_pos                         (1, 4, 50, 32) torch.float32 cuda:0
bp2_action_flat                                  (1, 4, 1600) torch.float32 cuda:0
bp2_action_embs                                  (1, 4, 2048) torch.float32 cuda:0
bp2_state_action_embs                            (1, 4, 4096) torch.float32 cuda:0
bp2_state_action_prompt_embs                     (1, 4, 2048) torch.bfloat16 cuda:0
bp2_prefix_embs                                  (1, 994, 2048) torch.bfloat16 cuda:0
bp2_prefix_pad_masks                             (1, 994) torch.bool cuda:0
bp2_pr

In [20]:
bp2_prefix_embs.shape

torch.Size([1, 994, 2048])

In [21]:
print("========== 4.3 full forward losses / 完整复刻 policy.forward 的三类 loss ==========")

# current images for middle branch: (B, num_views, T, C, H, W)
bp2_images = torch.stack(
    [
        bp_t_batch[f"{OBS_IMAGES}.image0"],
        bp_t_batch[f"{OBS_IMAGES}.image1"],
        bp_t_batch[f"{OBS_IMAGES}.image2"],
    ],
    dim=1,
)
bp2_img_masks = torch.ones(bp2_images.shape[:2], dtype=torch.bool, device=bp2_images.device)
bp2_state = bp_t_batch[OBS_STATE]
bp2_actions = bp_t_batch[ACTION]

# flow matching 与原 forward 一致。
bp2_noise = policy.model.sample_noise(bp2_actions.shape, bp2_actions.device)
bp2_time = policy.model.sample_time(bp2_actions.shape[0], bp2_actions.device)
bp2_u_t = bp2_noise - bp2_actions

with torch.no_grad():
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
        bp2_middle_embs, bp2_middle_pad_masks, bp2_middle_att_masks = policy.model.embed_middle(bp2_images, bp2_img_masks)
        bp2_suffix_embs, bp2_suffix_pad_masks, bp2_suffix_att_masks = policy.model.embed_suffix(bp2_state, bp2_noise, bp2_time)

if policy.model.qwen3_vl_with_expert.und_expert.language_model.layers[0].self_attn.q_proj.weight.dtype == torch.bfloat16:
    bp2_prefix_embs = bp2_prefix_embs.to(dtype=torch.bfloat16)
    bp2_middle_embs = bp2_middle_embs.to(dtype=torch.bfloat16)
    bp2_suffix_embs = bp2_suffix_embs.to(dtype=torch.bfloat16)

bp2_pad_masks = torch.cat([bp2_prefix_pad_masks, bp2_middle_pad_masks, bp2_suffix_pad_masks], dim=1)
bp2_att_masks = torch.cat([bp2_prefix_att_masks, bp2_middle_att_masks, bp2_suffix_att_masks], dim=1)
bp2_att_2d_masks = policy.model.build_training_attention_mask(
    bp2_pad_masks,
    bp2_att_masks,
    prefix_len=bp2_prefix_pad_masks.shape[1],
)
bp2_position_ids, bp2_rope_deltas = policy.model.get_position_ids(
    bp2_prefix_lang_tokens,
    bp2_image_grid_thw,
    bp2_pad_masks,
)
bp2_att_2d_masks_4d = policy.model._prepare_attention_masks_4d(bp2_att_2d_masks)

bp2_collect_middle_layers = policy.model.query_layer_indices if policy.model.da3_teacher is not None else None
with torch.no_grad():
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
        bp2_outputs = policy.model.qwen3_vl_with_expert.forward(
            attention_mask=bp2_att_2d_masks_4d,
            position_ids=bp2_position_ids,
            past_key_values=None,
            inputs_embeds=[bp2_prefix_embs, bp2_middle_embs, bp2_suffix_embs],
            use_cache=False,
            collect_middle_layers=bp2_collect_middle_layers,
        )

if bp2_collect_middle_layers is None:
    (_, bp2_middle_out, bp2_suffix_out), _ = bp2_outputs
    bp2_middle_layer_outputs = ()
else:
    (_, bp2_middle_out, bp2_suffix_out), _, bp2_middle_layer_outputs = bp2_outputs

# loss_gen：与原 forward 一致，用 middle visual tokens 重建未来帧 Cosmos latent。
if float(policy.config.lambda_gen) > 0.0:
    bp2_middle_visual_out, bp2_middle_query_out = policy.model.split_middle_tokens(bp2_middle_out)
    with torch.no_grad():
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            bp2_pred_cosmos_features = policy.model.decode_cosmos(bp2_middle_visual_out.to(dtype=torch.float32))
            bp2_future_embs = policy.model.get_cosmos_features(bp2_images[:, :, 2])
    bp2_loss_gen = F.mse_loss(
        bp2_pred_cosmos_features[bp2_img_masks],
        bp2_future_embs.to(dtype=torch.float32)[bp2_img_masks],
    )
else:
    bp2_middle_visual_out, bp2_middle_query_out = policy.model.split_middle_tokens(bp2_middle_out)
    bp2_pred_cosmos_features = None
    bp2_future_embs = None
    bp2_loss_gen = bp2_middle_out.new_zeros((), dtype=torch.float32)

# loss_3d：与原 forward 一致，直接复用模型内部函数。
bp2_loss_3d, bp2_loss_3d_logs = policy.model.compute_3d_query_loss(
    tuple(bp2_middle_layer_outputs),
    bp2_images[:, :, 2],
    bp2_img_masks,
)

# loss_action：与原 forward 一致，取 suffix 最后 chunk_size 个 action token。
bp2_suffix_out_action = bp2_suffix_out[:, -policy.config.chunk_size :].to(dtype=torch.float32)
bp2_v_t = policy.model.action_out_proj(bp2_suffix_out_action)
bp2_losses_action = F.mse_loss(bp2_u_t, bp2_v_t, reduction="none")

for name, value in [
    ("bp2_middle_out", bp2_middle_out),
    ("bp2_suffix_out", bp2_suffix_out),
    ("bp2_middle_visual_out", bp2_middle_visual_out),
    ("bp2_middle_query_out", bp2_middle_query_out),
    ("bp2_suffix_out_action", bp2_suffix_out_action),
    ("bp2_v_t", bp2_v_t),
    ("bp2_losses_action", bp2_losses_action),
    ("bp2_position_ids", bp2_position_ids),
]:
    print(f"{name:48s} {tuple(value.shape)} {value.dtype} {value.device}")

if bp2_pred_cosmos_features is not None:
    print(f"{'bp2_pred_cosmos_features':48s} {tuple(bp2_pred_cosmos_features.shape)} {bp2_pred_cosmos_features.dtype} {bp2_pred_cosmos_features.device}")
    print(f"{'bp2_future_embs':48s} {tuple(bp2_future_embs.shape)} {bp2_future_embs.dtype} {bp2_future_embs.device}")

print("bp2_loss_gen =", float(bp2_loss_gen.item()))
print("bp2_loss_3d =", float(bp2_loss_3d.item()))

========== 4.3 full forward losses / 完整复刻 policy.forward 的三类 loss ==========
[INFO ] Selecting reference view using strategy: saddle_balanced
bp2_middle_out                                   (1, 576, 1024) torch.bfloat16 cuda:0
bp2_suffix_out                                   (1, 51, 1024) torch.bfloat16 cuda:0
bp2_middle_visual_out                            (1, 144, 1024) torch.bfloat16 cuda:0
bp2_middle_query_out                             (1, 432, 1024) torch.bfloat16 cuda:0
bp2_suffix_out_action                            (1, 50, 1024) torch.float32 cuda:0
bp2_v_t                                          (1, 50, 32) torch.float32 cuda:0
bp2_losses_action                                (1, 50, 32) torch.float32 cuda:0
bp2_position_ids                                 (3, 1, 1621) torch.int64 cuda:0
bp2_pred_cosmos_features                         (1, 3, 16, 32, 32) torch.bfloat16 cuda:0
bp2_future_embs                                  (1, 3, 16, 32, 32) torch.float32 cuda:0
bp2_los

In [22]:
print("========== 4.4 full forward summary / 完整 forward 汇总 ==========")

bp2_losses_action_clipped = bp2_losses_action[:, :, : policy.config.max_action_dim]
bp2_action_loss_mask = bp_t_batch.get(
    SAMPLE_ACTION_LOSS_MASK,
    torch.ones(bp2_losses_action_clipped.shape[0], dtype=torch.bool, device=bp2_losses_action_clipped.device),
).to(dtype=torch.bool, device=bp2_losses_action_clipped.device)
bp2_action_loss_mask = bp2_action_loss_mask.reshape(bp2_losses_action_clipped.shape[0], -1).any(dim=1)

if bp2_action_loss_mask.any():
    bp2_sample_losses_action = bp2_losses_action_clipped[bp2_action_loss_mask]
    if policy.config.mask_action_dim_padding_loss:
        bp2_original_action_dim = policy.config.output_features["action"].shape[0]
        bp2_valid_action_dim = min(int(policy.config.action_loss_valid_dim), bp2_original_action_dim)
        bp2_loss_action = bp2_sample_losses_action[:, :, :bp2_valid_action_dim].mean()
    else:
        bp2_loss_action = bp2_sample_losses_action.mean()
else:
    bp2_loss_action = bp2_losses_action_clipped.new_zeros(())

bp2_loss = bp2_loss_action + policy.config.lambda_gen * bp2_loss_gen + policy.config.lambda_3d * bp2_loss_3d

bp2_forward_result = {
    "loss": float(bp2_loss.item()),
    "loss_action": float(bp2_loss_action.item()),
    "loss_gen": float(bp2_loss_gen.item()),
    "loss_3d": float(bp2_loss_3d.item()),
    "prefix_len": int(bp2_prefix_pad_masks.shape[1]),
    "middle_len": int(bp2_middle_pad_masks.shape[1]),
    "suffix_len": int(bp2_suffix_pad_masks.shape[1]),
    "total_len": int(bp2_pad_masks.shape[1]),
}
for key, value in bp2_loss_3d_logs.items():
    bp2_forward_result[key] = float(value.item())

for name, value in [
    ("bp2_losses_action_clipped", bp2_losses_action_clipped),
    ("bp2_action_loss_mask", bp2_action_loss_mask),
]:
    print(f"{name:48s} {tuple(value.shape)} {value.dtype} {value.device}")

print("bp2_loss        =", bp2_forward_result["loss"])
print("bp2_loss_action =", bp2_forward_result["loss_action"])
print("bp2_loss_gen    =", bp2_forward_result["loss_gen"])
print("bp2_loss_3d     =", bp2_forward_result["loss_3d"])
bp2_forward_result

========== 4.4 full forward summary / 完整 forward 汇总 ==========
bp2_losses_action_clipped                        (1, 50, 32) torch.float32 cuda:0
bp2_action_loss_mask                             (1,) torch.bool cuda:0
bp2_loss        = 0.7829182744026184
bp2_loss_action = 0.7508354187011719
bp2_loss_gen    = 2.1388673782348633
bp2_loss_3d     = 1.0694199800491333


{'loss': 0.7829182744026184,
 'loss_action': 0.7508354187011719,
 'loss_gen': 2.1388673782348633,
 'loss_3d': 1.0694199800491333,
 'prefix_len': 994,
 'middle_len': 576,
 'suffix_len': 51,
 'total_len': 1621,
 'time_3d_teacher_forward_s': 0.09709011018276215,
 'loss_3d_q13_t11': 0.822587788105011,
 'loss_3d_q19_t15': 1.2183455228805542,
 'loss_3d_q23_t19': 1.134506106376648,
 'loss_3d_q27_t23': 1.1022406816482544}